# Netflix Source Ingestion Testing

### Validate the structure of the real Netflix source files before loading them into PostgreSQL staging.

In [1]:
import sys

from pathlib import Path


# The notebook lives inside /notebooks, so the project root is one level above.
project_root = Path.cwd().parent


if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Display the detected root so we know the notebook is running from the correct project.
print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


## Inspect source file headers

Read only the CSV headers first.

This confirms that our staging schema matches the actual incoming source before any data is written to PostgreSQL.

In [2]:
import csv


# Point to the real titles source file.
titles_file = project_root / "data" / "raw" / "titles.csv"


# Point to the real credits source file.
credits_file = project_root / "data" / "raw" / "credits.csv"


# Confirm both expected files exist before continuing.
assert titles_file.exists(), f"Titles file not found: {titles_file}"
assert credits_file.exists(), f"Credits file not found: {credits_file}"


# Open the titles file and retrieve only the header row.
with titles_file.open(
    mode="r",
    encoding="utf-8",
    newline="",
) as source_file:

    # Create a CSV reader.
    reader = csv.reader(source_file)

    # Read only the first row, which contains the column names.
    titles_header = next(reader)


with credits_file.open(
    mode="r",
    encoding="utf-8",
    newline="",
) as source_file:

    reader = csv.reader(source_file)

    credits_header = next(reader)


# Display the actual source columns.
print("TITLES COLUMNS")
for column in titles_header:
    print("-", column)


print("\nCREDITS COLUMNS")
for column in credits_header:
    print("-", column)

TITLES COLUMNS
- id
- title
- type
- description
- release_year
- age_certification
- runtime
- genres
- production_countries
- seasons
- imdb_id
- imdb_score
- imdb_votes
- tmdb_popularity
- tmdb_score

CREDITS COLUMNS
- person_id
- id
- name
- character
- role


In [3]:
# Define the columns our staging design currently expects from titles.csv.
expected_titles_columns = [
    "id",
    "title",
    "type",
    "description",
    "release_year",
    "age_certification",
    "runtime",
    "genres",
    "production_countries",
    "seasons",
    "imdb_id",
    "imdb_score",
    "imdb_votes",
    "tmdb_popularity",
    "tmdb_score",
]


# Define the columns expected from credits.csv.
expected_credits_columns = [
    "person_id",
    "id",
    "name",
    "character",
    "role",
]


# Find expected title columns that are missing from the source.
missing_title_columns = [
    column
    for column in expected_titles_columns
    if column not in titles_header
]


# Find unexpected title columns supplied by the source.
extra_title_columns = [
    column
    for column in titles_header
    if column not in expected_titles_columns
]


missing_credit_columns = [
    column
    for column in expected_credits_columns
    if column not in credits_header
]


extra_credit_columns = [
    column
    for column in credits_header
    if column not in expected_credits_columns
]


# Display the comparison.
print("Missing title columns:", missing_title_columns)
print("Extra title columns:", extra_title_columns)

print("Missing credit columns:", missing_credit_columns)
print("Extra credit columns:", extra_credit_columns)

Missing title columns: []
Extra title columns: []
Missing credit columns: []
Extra credit columns: []


## Validate the real titles source

Read the actual titles source through the production ingestion module before writing anything to PostgreSQL.

In [4]:
# Import the production CSV validation function.
from src.ingestion import read_source_csv, TITLE_COLUMNS


# Read the real titles source using production code.
title_rows = read_source_csv(
    titles_file,
    TITLE_COLUMNS,
)


print("Source rows:", len(title_rows))


# Display the first record for inspection.
print("\nFirst record:")
print(title_rows[0])


# Confirm that the source is not empty.
assert len(title_rows) > 0


print("\nPASS: titles source validated successfully.")

Source rows: 12

First record:
{'id': 'ts_test_001', 'title': 'Harbour Lights', 'type': 'MOVIE', 'description': 'A retired sailor investigates a mystery in a coastal town.', 'release_year': '2021', 'age_certification': 'PG-13', 'runtime': '108', 'genres': "['drama', 'crime']", 'production_countries': "['GB']", 'seasons': '', 'imdb_id': 'tt9000001', 'imdb_score': '7.4', 'imdb_votes': '18420', 'tmdb_popularity': '18.6', 'tmdb_score': '7.1'}

PASS: titles source validated successfully.


## Test 3: Calculate the source fingerprint

Generate the real SHA-256 checksum used to identify this exact version of titles.csv.

In [ ]:
from src.file_utils import calculate_file_checksum


# Calculate the SHA-256 fingerprint of titles.csv.
titles_checksum = calculate_file_checksum(
    titles_file
)


# Display the fingerprint.
print("titles.csv SHA-256:")
print(titles_checksum)


# Confirm that SHA-256 produced a 64-character hexadecimal fingerprint.
assert len(titles_checksum) == 64


print("\nPASS: real source checksum generated.")

titles.csv SHA-256:
639cba13a200033e1ebb8e71243eef2d6d4453706ccff67e101f4935dbaad012

PASS: real source checksum generated.


## Check if the file has been processed before

In [ ]:
from src.batch_tracker import find_successful_batch_by_checksum


# Search the audit history for this exact source content.
existing_batch = find_successful_batch_by_checksum(
    titles_checksum
)


# Display the result.
print("Existing successful batch:", existing_batch)

Existing successful batch: None


## Register the real title batch 

In [7]:
# Import the production batch-registration function.
from src.batch_tracker import register_batch


# Register the real titles source before ingestion begins.
titles_batch_id = register_batch(
    pipeline_name="netflix_incremental_pipeline",
    file_name=titles_file.name,
    file_checksum=titles_checksum,
    rows_received=len(title_rows),
)


# Display the generated PostgreSQL batch identifier.
print("Titles batch ID:", titles_batch_id)


# Confirm that PostgreSQL generated an identifier.
assert titles_batch_id is not None


# Confirm successful registration.
print("PASS: real titles batch registered.")

Titles batch ID: 12
PASS: real titles batch registered.


## Mark the batch status as running

In [ ]:
from src.batch_tracker import mark_batch_running


# Mark this real batch as actively processing.
mark_batch_running(titles_batch_id)


# Confirm the transition was sent successfully.
print(
    f"Batch {titles_batch_id} moved to RUNNING."
)

Batch 12 moved to RUNNING.


## Load titles into PostgreSQL staging

Load the validated real source into staging.titles_raw using the registered batch ID.

In [ ]:
# Import the production staging loader.
from src.ingestion import load_titles_to_staging


# Load the real titles source into PostgreSQL staging.
staged_rows = load_titles_to_staging(
    file_path=titles_file,
    batch_id=titles_batch_id,
)


# Display how many records PostgreSQL received.
print("Rows staged:", staged_rows)


# Source count and staging-loader count must agree.
assert staged_rows == len(title_rows)


print("PASS: titles.csv loaded into staging.")

Rows staged: 5850
PASS: titles.csv loaded into staging.


## Validate the real credits source

Read the actual credits CSV through the production ingestion module before writing it to PostgreSQL.

In [ ]:
# Import the production credits schema definition.
from src.ingestion import read_source_csv, CREDIT_COLUMNS


credit_rows = read_source_csv(
    credits_file,
    CREDIT_COLUMNS,
)

print("Credits source rows:", len(credit_rows))


# Show one record for inspection.
print("\nFirst credit record:")
print(credit_rows[0])


# Confirm that the source is not empty.
assert len(credit_rows) > 0

print("\nPASS: credits source validated successfully.")

Credits source rows: 77801

First credit record:
{'person_id': '3748', 'id': 'tm84618', 'name': 'Robert De Niro', 'character': 'Travis Bickle', 'role': 'ACTOR'}

PASS: credits source validated successfully.


## Calculate credits checksum

Generate the SHA-256 fingerprint for credits.csv so duplicate batches can be detected.

In [11]:
# Generate the real checksum for credits.csv.
credits_checksum = calculate_file_checksum(
    credits_file
)


# Display the checksum.
print("credits.csv SHA-256:")
print(credits_checksum)


# SHA-256 hexadecimal output should contain 64 characters.
assert len(credits_checksum) == 64


# Confirm successful checksum generation.
print("\nPASS: credits checksum generated.")

credits.csv SHA-256:
7122fdc347134241a3192436ae1823ed3e2a50e57985e10ee337cd86b260d831

PASS: credits checksum generated.


## Check for duplicates in batch history

In [12]:
# Search for an already successful batch with identical file content.
existing_credits_batch = find_successful_batch_by_checksum(
    credits_checksum
)


# Display the duplicate-check result.
print(
    "Existing successful credits batch:",
    existing_credits_batch,
)

Existing successful credits batch: None


## Register the real credits batch

In [13]:
# Register credits.csv before processing begins.
credits_batch_id = register_batch(
    pipeline_name="netflix_incremental_pipeline",
    file_name=credits_file.name,
    file_checksum=credits_checksum,
    rows_received=len(credit_rows),
)


# Display the generated batch identifier.
print("Credits batch ID:", credits_batch_id)


# Confirm a batch ID was generated.
assert credits_batch_id is not None


# Confirm successful registration.
print("PASS: real credits batch registered.")

Credits batch ID: 13
PASS: real credits batch registered.


## Mark batch status as Running

In [14]:
# Mark the credits batch as actively processing.
mark_batch_running(
    credits_batch_id
)


# Display confirmation.
print(
    f"Batch {credits_batch_id} moved to RUNNING."
)

Batch 13 moved to RUNNING.


## Load credits into PostgreSQL staging

Load the validated credits source into staging.credits_raw.

In [ ]:
# Import the production credits staging loader.
from src.ingestion import load_credits_to_staging


# Load the real credits source into staging.
credits_staged_rows = load_credits_to_staging(
    file_path=credits_file,
    batch_id=credits_batch_id,
)


# Display the staging count returned by the loader.
print(
    "Credits rows staged:",
    credits_staged_rows,
)


# Source count and staging-loader count must match.
assert credits_staged_rows == len(credit_rows)


print(
    "PASS: credits.csv loaded into staging."
)

Credits rows staged: 77801
PASS: credits.csv loaded into staging.


## Verify the physical PostgresSQL count

In [ ]:
from src.database import get_etl_connection


with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count the rows physically stored for this batch.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM staging.credits_raw
            WHERE batch_id = %s;
            """,
            (credits_batch_id,),
        )

        # Retrieve the count.
        credits_database_row_count = cursor.fetchone()[0]


# Display source and database counts.
print(
    "Credits CSV rows:     ",
    len(credit_rows),
)

print(
    "Credits database rows:",
    credits_database_row_count,
)


# Require an exact match.
assert credits_database_row_count == len(credit_rows)


# Confirm the quality gate.
print(
    "PASS: credits source and staging row counts match."
)

Credits CSV rows:      77801
Credits database rows: 77801
PASS: credits source and staging row counts match.


## Combined staging verification

In [ ]:
# Open one final verification connection.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count staged title rows for the real titles batch.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM staging.titles_raw
            WHERE batch_id = %s;
            """,
            (titles_batch_id,),
        )

        titles_count = cursor.fetchone()[0]

        # Count staged credit rows for the real credits batch.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM staging.credits_raw
            WHERE batch_id = %s;
            """,
            (credits_batch_id,),
        )

        credits_count = cursor.fetchone()[0]


# Display the staging summary.
print("Titles staged:", titles_count)
print("Credits staged:", credits_count)


# Confirm both staging areas contain data.
assert titles_count > 0
assert credits_count > 0


print(
    "PASS: both real Netflix sources are available in staging."
)

Titles staged: 5850
Credits staged: 77801
PASS: both real Netflix sources are available in staging.


## Validate Automatic Source Ingestion

Test the production source-batch runner that will replace manual notebook
ingestion.

The runner detects a complete `titles.csv` and `credits.csv` pair, validates
their structure, prevents duplicate ingestion using checksums, registers both
batches, and loads them into staging.

When no incoming files exist, the runner should return no work rather than
treating the condition as a pipeline failure.

In [5]:
import importlib

import src.batch_tracker as batch_tracker
import src.source_batch_runner as source_batch_runner


# Reload the modules so the notebook uses the latest production code.
importlib.reload(
    batch_tracker
)

importlib.reload(
    source_batch_runner
)


ingest_incoming_batch = (
    source_batch_runner.ingest_incoming_batch
)


print(
    "PASS: automatic ingestion components loaded."
)

PASS: automatic ingestion components loaded.


In [6]:
# No source files currently exist in data/incoming/.
result = ingest_incoming_batch()

print(
    "Ingestion result:",
    result,
)

assert result is None

print(
    "PASS: empty incoming directory handled correctly."
)

Ingestion result: None
PASS: empty incoming directory handled correctly.


In [8]:
ingestion_result = ingest_incoming_batch()

print(
    ingestion_result
)

{'titles_batch_id': 14, 'credits_batch_id': 15, 'titles_rows': 12, 'credits_rows': 34}
